In [3]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [4]:
from bs4 import BeautifulSoup
import pandas as pd

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 정보가 포함된 li 태그 찾기
    for paper in soup.find_all('li', class_='entry'):
        # 제목 찾기
        title_tag = paper.find('span', class_='title')
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        author_tags = paper.find_all('span', itemprop='author')
        authors_cleaned = ", ".join([author.text.strip() for author in author_tags]) if author_tags else "Unknown"

        # PDF 링크 만들기
        pdf_link = None
        
        # ACL Anthology 링크 찾기
        aclanthology_link = paper.find('a', href=True)
        if aclanthology_link and "aclanthology.org" in aclanthology_link['href']:
            pdf_link = aclanthology_link['href'] + ".pdf"
        
        # DOI 링크 찾기
        elif aclanthology_link and "doi.org" in aclanthology_link['href']:
            # DOI 링크에서 논문 코드만 추출
            doi_link = aclanthology_link['href']
            paper_code = doi_link.split('/')[-1]  # DOI에서 논문 코드만 추출
            pdf_link = f"https://aclanthology.org/{paper_code}.pdf"
        
        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [5]:
url = 'https://dblp.org/db/conf/emnlp/emnlp2019-1.html'
DB_PATH = "con_db/EMNLP_conference_2019.db"
conference_name = 'EMNLP 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [6]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/EMNLP_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [7]:
df_papers = get_www_papers('html/EMNLP_2019_accepted_papers.html', conference_name)

In [8]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 2019 Conference on Empirica...,"Kentaro Inui, Jing Jiang, Vincent Ng, Xiaojun Wan",https://aclanthology.org/volumes/D19-1/.pdf,None,EMNLP 2019
1,Attending to Future Tokens for Bidirectional S...,"Carolin Lawrence, Bhushan Kotnis, Mathias Niepert",https://aclanthology.org/D19-1001.pdf,None,EMNLP 2019
2,Attention is not not Explanation.,"Sarah Wiegreffe, Yuval Pinter",https://aclanthology.org/D19-1002.pdf,None,EMNLP 2019
3,Practical Obstacles to Deploying Active Learning.,"David Lowell, Zachary C. Lipton, Byron C. Wallace",https://aclanthology.org/D19-1003.pdf,None,EMNLP 2019
4,Transfer Learning Between Related Tasks Using ...,"Matan Ben Noach, Yoav Goldberg",https://aclanthology.org/D19-1004.pdf,None,EMNLP 2019


In [9]:
df_papers = df_papers.drop(index=[0])

In [10]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Attending to Future Tokens for Bidirectional S...,"Carolin Lawrence, Bhushan Kotnis, Mathias Niepert",https://aclanthology.org/D19-1001.pdf,None,EMNLP 2019
2,Attention is not not Explanation.,"Sarah Wiegreffe, Yuval Pinter",https://aclanthology.org/D19-1002.pdf,None,EMNLP 2019
3,Practical Obstacles to Deploying Active Learning.,"David Lowell, Zachary C. Lipton, Byron C. Wallace",https://aclanthology.org/D19-1003.pdf,None,EMNLP 2019
4,Transfer Learning Between Related Tasks Using ...,"Matan Ben Noach, Yoav Goldberg",https://aclanthology.org/D19-1004.pdf,None,EMNLP 2019
5,Knowledge Enhanced Contextual Word Representat...,"Matthew E. Peters, Mark Neumann, Robert L. Log...",https://aclanthology.org/D19-1005.pdf,None,EMNLP 2019


In [11]:
save_to_database(df_papers, conference_name, DB_PATH)

681개의 논문이 EMNLP 2019에 저장되었습니다.


# 2018

In [50]:
from bs4 import BeautifulSoup
import pandas as pd

def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 정보가 포함된 li 태그 찾기
    for paper in soup.find_all('li', class_='entry'):
        # 제목 찾기
        title_tag = paper.find('span', class_='title')
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        author_tags = paper.find_all('span', itemprop='author')
        authors_cleaned = ", ".join([author.text.strip() for author in author_tags]) if author_tags else "Unknown"

        # PDF 링크 만들기
        pdf_link = None
        
        # ACL Anthology 링크 찾기
        aclanthology_link = paper.find('a', href=True)
        if aclanthology_link and "aclanthology.org" in aclanthology_link['href']:
            pdf_link = aclanthology_link['href'] + ".pdf"
        
        # DOI 링크 찾기
        elif aclanthology_link and "doi.org" in aclanthology_link['href']:
            # DOI 링크에서 논문 코드만 추출
            doi_link = aclanthology_link['href']
            paper_code = doi_link.split('/')[-1]  # DOI에서 논문 코드만 추출
            paper_code = paper_code[0].upper() + paper_code[1:]  # 첫 글자를 대문자로 변환
            pdf_link = f"https://aclanthology.org/{paper_code}.pdf"
        
        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers


In [51]:
url = 'https://dblp.org/db/conf/emnlp/emnlp2018.html'
DB_PATH = "con_db/EMNLP_conference_2018.db"
conference_name = 'EMNLP 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [52]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/EMNLP_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [53]:
df_papers = get_www_papers('html/EMNLP_2018_accepted_papers.html', conference_name)

In [54]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 2018 Conference on Empirica...,"Ellen Riloff, David Chiang, Julia Hockenmaier,...",https://aclanthology.org/volumes/D18-1/.pdf,None,EMNLP 2018
1,Privacy-preserving Neural Representations of T...,"Maximin Coavoux, Shashi Narayan, Shay B. Cohen",https://aclanthology.org/D18-1001.pdf,None,EMNLP 2018
2,Adversarial Removal of Demographic Attributes ...,"Yanai Elazar, Yoav Goldberg",https://aclanthology.org/D18-1002.pdf,None,EMNLP 2018
3,DeClarE: Debunking Fake News and False Claims ...,"Kashyap Popat, Subhabrata Mukherjee, Andrew Ya...",https://aclanthology.org/D18-1003.pdf,None,EMNLP 2018
4,It's going to be okay: Measuring Access to Sup...,"Zijian Wang, David Jurgens",https://aclanthology.org/D18-1004.pdf,None,EMNLP 2018


In [55]:
df_papers = df_papers.drop(index=[0])

In [56]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Privacy-preserving Neural Representations of T...,"Maximin Coavoux, Shashi Narayan, Shay B. Cohen",https://aclanthology.org/D18-1001.pdf,None,EMNLP 2018
2,Adversarial Removal of Demographic Attributes ...,"Yanai Elazar, Yoav Goldberg",https://aclanthology.org/D18-1002.pdf,None,EMNLP 2018
3,DeClarE: Debunking Fake News and False Claims ...,"Kashyap Popat, Subhabrata Mukherjee, Andrew Ya...",https://aclanthology.org/D18-1003.pdf,None,EMNLP 2018
4,It's going to be okay: Measuring Access to Sup...,"Zijian Wang, David Jurgens",https://aclanthology.org/D18-1004.pdf,None,EMNLP 2018
5,Detecting Gang-Involved Escalation on Social M...,"Serina Chang, Ruiqi Zhong, Ethan Adams, Fei-Tz...",https://aclanthology.org/D18-1005.pdf,None,EMNLP 2018


In [57]:
save_to_database(df_papers, conference_name, DB_PATH) 

549개의 논문이 EMNLP 2018에 저장되었습니다.


# 2017

In [58]:
url = 'https://dblp.org/db/conf/emnlp/emnlp2017.html'
DB_PATH = "con_db/EMNLP_conference_2017.db"
conference_name = 'EMNLP 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [59]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/EMNLP_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [60]:
df_papers = get_www_papers('html/EMNLP_2017_accepted_papers.html', conference_name)

In [61]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 2017 Conference on Empirica...,"Martha Palmer, Rebecca Hwa, Sebastian Riedel",https://aclanthology.org/volumes/D17-1/.pdf,None,EMNLP 2017
1,Monolingual Phrase Alignment on Parse Forests.,"Yuki Arase, Junichi Tsujii",https://aclanthology.org/D17-1001.pdf,None,EMNLP 2017
2,Fast(er) Exact Decoding and Global Training fo...,"Tianze Shi, Liang Huang, Lillian Lee",https://aclanthology.org/D17-1002.pdf,None,EMNLP 2017
3,Quasi-Second-Order Parsing for 1-Endpoint-Cros...,"Junjie Cao, Sheng Huang, Weiwei Sun, Xiaojun Wan",https://aclanthology.org/D17-1003.pdf,None,EMNLP 2017
4,Position-aware Attention and Supervised Data I...,"Yuhao Zhang, Victor Zhong, Danqi Chen, Gabor A...",https://aclanthology.org/D17-1004.pdf,None,EMNLP 2017


In [62]:
df_papers = df_papers.drop(index=[0])

In [63]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Monolingual Phrase Alignment on Parse Forests.,"Yuki Arase, Junichi Tsujii",https://aclanthology.org/D17-1001.pdf,None,EMNLP 2017
2,Fast(er) Exact Decoding and Global Training fo...,"Tianze Shi, Liang Huang, Lillian Lee",https://aclanthology.org/D17-1002.pdf,None,EMNLP 2017
3,Quasi-Second-Order Parsing for 1-Endpoint-Cros...,"Junjie Cao, Sheng Huang, Weiwei Sun, Xiaojun Wan",https://aclanthology.org/D17-1003.pdf,None,EMNLP 2017
4,Position-aware Attention and Supervised Data I...,"Yuhao Zhang, Victor Zhong, Danqi Chen, Gabor A...",https://aclanthology.org/D17-1004.pdf,None,EMNLP 2017
5,Heterogeneous Supervision for Relation Extract...,"Liyuan Liu, Xiang Ren, Qi Zhu, Shi Zhi, Huan G...",https://aclanthology.org/D17-1005.pdf,None,EMNLP 2017


In [64]:
save_to_database(df_papers, conference_name, DB_PATH)

322개의 논문이 EMNLP 2017에 저장되었습니다.


# 2016

In [65]:
url = 'https://dblp.org/db/conf/emnlp/emnlp2016.html'
DB_PATH = "con_db/EMNLP_conference_2016.db"
conference_name = 'EMNLP 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [66]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/EMNLP_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [67]:
df_papers = get_www_papers('html/EMNLP_2016_accepted_papers.html', conference_name)

In [68]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 2016 Conference on Empirica...,"Jian Su, Xavier Carreras, Kevin Duh",https://aclanthology.org/volumes/D16-1/.pdf,None,EMNLP 2016
1,Span-Based Constituency Parsing with a Structu...,"James Cross, Liang Huang",https://aclanthology.org/D16-1001.pdf,None,EMNLP 2016
2,Rule Extraction for Tree-to-Tree Transducers b...,"Pascual Martínez-Gómez, Yusuke Miyao",https://aclanthology.org/D16-1002.pdf,None,EMNLP 2016
3,A Neural Network for Coordination Boundary Pre...,"Jessica Ficler, Yoav Goldberg",https://aclanthology.org/D16-1003.pdf,None,EMNLP 2016
4,Using Left-corner Parsing to Encode Universal ...,"Hiroshi Noji, Yusuke Miyao, Mark Johnson",https://aclanthology.org/D16-1004.pdf,None,EMNLP 2016


In [69]:
df_papers = df_papers.drop(index=[0])

In [70]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Span-Based Constituency Parsing with a Structu...,"James Cross, Liang Huang",https://aclanthology.org/D16-1001.pdf,None,EMNLP 2016
2,Rule Extraction for Tree-to-Tree Transducers b...,"Pascual Martínez-Gómez, Yusuke Miyao",https://aclanthology.org/D16-1002.pdf,None,EMNLP 2016
3,A Neural Network for Coordination Boundary Pre...,"Jessica Ficler, Yoav Goldberg",https://aclanthology.org/D16-1003.pdf,None,EMNLP 2016
4,Using Left-corner Parsing to Encode Universal ...,"Hiroshi Noji, Yusuke Miyao, Mark Johnson",https://aclanthology.org/D16-1004.pdf,None,EMNLP 2016
5,"Distinguishing Past, On-going, and Future Even...","Ruihong Huang, Ignacio Cases, Dan Jurafsky, Cl...",https://aclanthology.org/D16-1005.pdf,None,EMNLP 2016


In [71]:
save_to_database(df_papers, conference_name, DB_PATH) 

264개의 논문이 EMNLP 2016에 저장되었습니다.


In [72]:
url = 'https://dblp.org/db/conf/emnlp/emnlp2015.html'
DB_PATH = "con_db/EMNLP_conference_2015.db"
conference_name = 'EMNLP 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [73]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/EMNLP_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [74]:
df_papers = get_www_papers('html/EMNLP_2015_accepted_papers.html', conference_name)

In [75]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 2015 Conference on Empirica...,"Lluís Màrquez, Chris Callison-Burch, Jian Su, ...",https://aclanthology.org/volumes/D15-1/.pdf,None,EMNLP 2015
1,Language Understanding for Text-based Games us...,"Karthik Narasimhan, Tejas D. Kulkarni, Regina ...",https://aclanthology.org/D15-1001.pdf,None,EMNLP 2015
2,Distributional vectors encode referential attr...,"Abhijeet Gupta, Gemma Boleda, Marco Baroni, Se...",https://aclanthology.org/D15-1002.pdf,None,EMNLP 2015
3,Building a shared world: mapping distributiona...,"Aurélie Herbelot, Eva Maria Vecchi",https://aclanthology.org/D15-1003.pdf,None,EMNLP 2015
4,Dependency Graph-to-String Translation.,"Liangyou Li, Andy Way, Qun Liu",https://aclanthology.org/D15-1004.pdf,None,EMNLP 2015


In [76]:
df_papers = df_papers.drop(index=[0])

In [77]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Language Understanding for Text-based Games us...,"Karthik Narasimhan, Tejas D. Kulkarni, Regina ...",https://aclanthology.org/D15-1001.pdf,None,EMNLP 2015
2,Distributional vectors encode referential attr...,"Abhijeet Gupta, Gemma Boleda, Marco Baroni, Se...",https://aclanthology.org/D15-1002.pdf,None,EMNLP 2015
3,Building a shared world: mapping distributiona...,"Aurélie Herbelot, Eva Maria Vecchi",https://aclanthology.org/D15-1003.pdf,None,EMNLP 2015
4,Dependency Graph-to-String Translation.,"Liangyou Li, Andy Way, Qun Liu",https://aclanthology.org/D15-1004.pdf,None,EMNLP 2015
5,Reordering Grammar Induction.,"Milos Stanojevic, Khalil Sima'an",https://aclanthology.org/D15-1005.pdf,None,EMNLP 2015


In [78]:
save_to_database(df_papers, conference_name, DB_PATH)

312개의 논문이 EMNLP 2015에 저장되었습니다.


# 2014

In [79]:
url = 'https://dblp.org/db/conf/emnlp/emnlp2014.html'
DB_PATH = "con_db/EMNLP_conference_2014.db"
conference_name = 'EMNLP 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [80]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/EMNLP_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [81]:
df_papers = get_www_papers('html/EMNLP_2014_accepted_papers.html', conference_name)

In [82]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the 2014 Conference on Empirica...,"Alessandro Moschitti, Bo Pang, Walter Daelemans",https://aclanthology.org/volumes/D14-1/.pdf,None,EMNLP 2014
1,Invited Talk: IBM Cognitive Computing - An NLP...,Salim Roukos,https://aclanthology.org/D14-1001.pdf,None,EMNLP 2014
2,Modeling Interestingness with Deep Neural Netw...,"Jianfeng Gao, Patrick Pantel, Michael Gamon, X...",https://aclanthology.org/D14-1002.pdf,None,EMNLP 2014
3,Translation Modeling with Bidirectional Recurr...,"Martin Sundermeyer, Tamer Alkhouli, Joern Wueb...",https://aclanthology.org/D14-1003.pdf,None,EMNLP 2014
4,A Neural Network Approach to Selectional Prefe...,Tim Van de Cruys,https://aclanthology.org/D14-1004.pdf,None,EMNLP 2014


In [83]:
df_papers = df_papers.drop(index=[0])

In [84]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Invited Talk: IBM Cognitive Computing - An NLP...,Salim Roukos,https://aclanthology.org/D14-1001.pdf,None,EMNLP 2014
2,Modeling Interestingness with Deep Neural Netw...,"Jianfeng Gao, Patrick Pantel, Michael Gamon, X...",https://aclanthology.org/D14-1002.pdf,None,EMNLP 2014
3,Translation Modeling with Bidirectional Recurr...,"Martin Sundermeyer, Tamer Alkhouli, Joern Wueb...",https://aclanthology.org/D14-1003.pdf,None,EMNLP 2014
4,A Neural Network Approach to Selectional Prefe...,Tim Van de Cruys,https://aclanthology.org/D14-1004.pdf,None,EMNLP 2014
5,Learning Image Embeddings using Convolutional ...,"Douwe Kiela, Léon Bottou",https://aclanthology.org/D14-1005.pdf,None,EMNLP 2014


In [85]:
save_to_database(df_papers, conference_name, DB_PATH)

226개의 논문이 EMNLP 2014에 저장되었습니다.
